# Part 1 — Scope, Ownership, and Outputs

### Objective

Build the local analysis-ready dataset for dynamic AKI prediction. This notebook owns source validation, cohort construction, KDIGO onset timelines, hourly snapshots, multi-horizon labels, leakage-safe features, and patient-level splits.

### Outputs

- Eligible ICU cohort
- Dataset-local patient/stay identifier crosswalk
- Hourly snapshot dataset
- Patient-level split manifest
- Feature dictionary and aggregate quality report

### Rules

Do not train models here, upload MIMIC data, or commit patient-level artifacts.


# Part 2 — Configuration and Study Definitions

### Objective

Load all study choices from configuration before processing data.

### Required settings

- Local MIMIC-III and artifact roots
- Monitoring start and end times
- Training snapshot interval: 1 hour
- Prediction horizons: 6, 12, 24, and 48 hours
- Operational horizon: selected on validation data
- Feature lookback windows
- Follow-up and exclusion rules
- Patient split proportions and random seed
- AKI and baseline-creatinine definition versions

### Rule

Do not redefine these choices in later cells without creating a new dataset version.


# Part 3 — Load and Validate MIMIC-III Tables

### Objective

Load only required columns and standardize identifiers, timestamps, units, and missing values.

### Candidate sources

PATIENTS, ADMISSIONS, ICUSTAYS, LABEVENTS, CHARTEVENTS, OUTPUTEVENTS, and approved diagnosis, procedure, medication, and intervention tables.

### Checks

- subject_id, hadm_id, and icustay_id relationships
- One-to-one mapping of dataset-local patient_id to source subject_id
- Source counts and timestamp ranges
- CareVue and MetaVision item mappings
- Duplicate, invalid, and incompatible-unit measurements

### Output

Validated local source views with explicit schemas and an identifier crosswalk.


# Part 4 — Define the Base ICU Cohort

### Objective

Create the eligible population before repeated snapshots are generated.

### Inclusion rules

- Adults under the prespecified MIMIC age rule
- Exactly one selected ICU stay per patient, chosen by a prespecified eligibility/chronology rule
- Sufficient follow-up for at least one prediction horizon

### Exclusion rules

- ESKD or chronic dialysis under prespecified definitions
- Invalid ICU timing
- Additional exclusions documented before use

### Output

One row per eligible patient and selected ICU stay, including patient_id, subject_id, hadm_id, icustay_id, and the selection-rule version.


# Part 5 — Define Baseline Creatinine and Renal History

### Objective

Create a chronological creatinine record and assign the primary baseline used for KDIGO assessment.

### Rules

- Standardize units and preserve measurement provenance.
- Define how pre-ICU, admission, nadir, and missing baselines are handled.
- Do not use future measurements to construct a snapshot's input features.
- Record CKD, prior renal replacement therapy, and exclusion evidence separately.

### Sensitivity plan

Prepare alternative defensible baseline definitions for later robustness analysis rather than silently choosing the most favorable result.


# Part 6 — Generate KDIGO AKI Onset Timelines

### Objective

Identify the earliest AKI onset, stage, and supporting criterion.

### Primary definition

Creatinine increase of at least 0.3 mg/dL within 48 hours or at least 1.5 times the applicable baseline within the prespecified KDIGO period.

### Robustness extension

Add urine-output KDIGO using documented body weight, rolling windows, missing-output handling, and renal replacement therapy rules.

### Checks

Use manually constructed positive, negative, boundary, duplicate-time, missing-baseline, and low-urine-output cases.


# Part 7 — Freeze Patient Identity and Development Splits

### Objective

Create the immutable one-patient/one-selected-stay cohort and assign each patient to exactly one reproducible development split before generating any snapshots, labels, or features.

### Rules

- Create a dataset-local `patient_id` surrogate with a one-to-one mapping to source `subject_id`; retain `hadm_id` and `icustay_id` as provenance keys.
- Select exactly one eligible ICU stay per patient using a prespecified rule based only on eligibility and source chronology.
- Write one split-manifest row per patient/selected stay.
- Assign all future rows for that patient to the inherited train, validation, or test split.
- Stop before proceeding if any patient has multiple selected stays, a missing split, or a split overlap.

### Output

One row per selected patient/stay in the frozen cohort and split manifest.


# Part 8 — Generate Hourly Training Snapshots

### Objective

Create one candidate prediction cutoff per selected ICU stay and ICU hour during the configured monitoring period.

### Rules

- Inherit `patient_id`, `subject_id`, `hadm_id`, `icustay_id`, and the preassigned split from the frozen manifest.
- feature_time <= snapshot_time.
- Stop snapshots at AKI onset, ICU discharge, death, or monitoring end.
- Keep actual AKI-relevant EHR event times for later event-driven replay.
- Do not create minute-level duplicate training rows when no relevant data changed.

### Output

One candidate row per selected `icustay_id` and `snapshot_time`, with inherited patient and split keys.


# Part 9 — Assign 6h, 12h, 24h, and 48h AKI Targets

### Objective

For every snapshot, assign horizon-specific eligibility and labels.

### Label rule

For horizon H, the positive window is (snapshot_time, snapshot_time + H]. A patient with AKI at or before snapshot_time is not at risk and must not contribute a later snapshot.

### Required fields

- eligible_6h, aki_within_6h
- eligible_12h, aki_within_12h
- eligible_24h, aki_within_24h
- eligible_48h, aki_within_48h
- aki_onset_time when applicable

Overlapping labels across hours and horizons are expected and are not leakage.

### Next gate

Labels are created after the cohort and split are frozen and before feature extraction.


# Part 10 — Extract Leakage-Safe Multimodal Features

### Objective

Summarize information available by each snapshot.

### Modalities

- Demographics and comorbidities
- Vital signs and laboratory measurements
- Glasgow Coma Scale
- Urine output
- Medications and interventions when feasible

### Summaries

Use prespecified latest, mean, minimum, maximum, range, slope, variability, count, missingness, and time-since-last-measurement features over clinically meaningful windows.

### Caution

Measurement frequency can encode workflow and illness severity; retain it only when intentional and test its transportability.


# Part 11 — Build the Dynamic Modeling Table

### Objective

Join all snapshot-specific features and targets into a stable schema while preserving the frozen patient/stay identity and split assignment.

### Required fields

- patient_id, subject_id, hadm_id, and icustay_id
- split, snapshot_time, and continuous hours_since_icu
- Horizon-specific labels and eligibility flags
- Feature columns and provenance version
- CareVue or MetaVision source indicator for robustness analysis

### Rules

Keep identifiers, outcomes, onset times, split columns, and future information out of the model feature list. Preserve 6h, 12h, 24h, and 48h as horizon-specific targets.

### Temporal generalization note

Every snapshot from a patient must inherit the same split; never re-split this table by row. MIMIC-III calendar years are patient-shifted and must not be treated as true chronology.


# Part 12 — Validate and Export Dataset Artifacts

### Objective

Run final integrity checks and write versioned local artifacts.

### Required checks

- One-to-one patient_id to subject_id mapping
- One selected icustay_id per patient_id
- Patient disjointness across splits
- No feature event after its snapshot
- No snapshot at or after AKI onset
- Correct horizon boundaries and follow-up eligibility
- Unique snapshot keys
- Outcome prevalence and sample counts by split, hour, horizon, ICU type, and source system
- Reconciliation of horizon-specific outcome prevalence and known cohort counts

### Outputs

Export the cohort, snapshot dataset, split manifest, feature dictionary, configuration, and aggregate quality report. Never commit patient-level outputs.
